# Connecting the MCP Chatbot to Reference Servers 

You'll now extend the MCP chatbot capabilities by making it connect to any MCP server. You will integrate the tools of two official MCP servers in addition to the tools of the research server you built in a previous lesson. 

<img src="images/lesson_6.png" width="400">

## Open-Source MCP Servers

In this [repo](https://github.com/modelcontextprotocol/servers), you can find a collection of reference implementations for the MCP servers, as well as references to community built servers and additional resources. You will use two of the reference servers to integrate their tools in your MCP chatbot:
- [fetch](https://github.com/modelcontextprotocol/servers/tree/main/src/fetch): provides the `fetch` tool which fetches a URL from the internet and extracts its contents as markdown.
- [filesystem](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem): provides several tools for interacting with the files and directories within a directory that you specify.

You can check the readme file of each server to check the features they expose and how to run them.  

## Updating the MCP Chatbot - Optional Reading

- Instead of hardcoding the server parameters in the chatbot, the chatbot will read the server configurations from a JSON file:
  ### Server Configuration
  In the `mcp_project`, you can find the `server_config.json` configuration file that has the following structure.
    ``` json
    {
        "mcpServers": {
            
            "filesystem": {
                "command": "npx",
                "args": [
                    "-y",
                    "@modelcontextprotocol/server-filesystem",
                    "."
                ]
            },
            
            "research": {
                "command": "python",
                "args": ["research_server.py"]
            },
            
            "fetch": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-fetch"]
            }
        }
    }
    ```
    For the reference servers, `npx` directly installs and runs the servers without needing to install them ahead of time. Note for the `filesystem`, the `.` means "current directory" — you're allowing the server to interact with files in the current directory.


## Requirements

To run this notebook, you need **`uv`** and **`uvx`** installed on your system.

`uv` is a fast Python package manager, and `uvx` is included automatically when you install `uv`.

---

## Installation

Please follow the official installation guide:

https://docs.astral.sh/uv/getting-started/installation/#standalone-installer

---

## Windows Installation (Quick Steps)

### 1️⃣ Install using PowerShell

Open **PowerShell** and run:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

### 2️⃣ Add uv to Environment Variables

After installation, add the following path to your User PATH environment variable:

```
C:\Users\YOUR_USERNAME\.local\bin
```

### Verify Installation

After restarting PowerShell, run:
```uv --version```

- Here's a rough sketch of the diagram of the updated MCP_chatbot:
  
  <img src="images/updated_class.png" width="1000">

  1. Instead of having one session, you now have a list of client sessions where each client session establishes a 1-to-1 connection to each server;
  2. `available_tools` includes the definitions of all the tools exposed by all servers that the chatbot can connect to.
  3. `tool_to_session` maps the tool name to the corresponding client session; in this way, when the LLM decides on a particular tool name, you can map it to the correct client session so you can use that session to send `tool_call` request to the right MCP server.
  4. `exit_stack` is a context manager that will manage the mcp client objects and their sessions and ensures that they are properly closed. In the previous lesson , you did not use it because you used the `with` statement which behind the scenes uses a context manager. Here you could again use the `with` statement, but you may end up using multiple nested `with` statements since you have multiple servers to connect to. `exit_stack` allows you to dynamically add the mcp clients and their sessions as you'll see in the code below.
  5. `connect_to_servers` reads the server configuration file and for each single server, it calls the helper method `connect_to_server`. In this latter method, an MCP client is created and used to launch the server as a sub-process and then a client session is created to connect to the server and get a description of the list of the tools provided by the server.
  6. `cleanup` is a helper method that ensures all your connections are properly shut down when you're done with them. In the previous lesson, you relied on the `with` statement to automatically clean up resources. This cleanup method serves a similar purpose, but for all the resources you've added to your exit_stack; it closes (your MCP clients and sessions) in the reverse order they were added - like stacking and unstacking plates. This is particularly important in network programming to avoid resource leaks.

## Updated Code for the MCP Chatbot

In [ ]:
%%writefile mcp_project/new_mcp_chatbot.py

from dotenv import load_dotenv
from openai import OpenAI
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from typing import List, Dict, TypedDict
from contextlib import AsyncExitStack
import json
import asyncio
import os

# Load environment variables such as the OpenAI API key.
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")
model_base = "gpt-5.1"


# Define the expected structure of each tool in OpenAI function-calling format.
class ToolDefinition(TypedDict):
    type: str
    function: dict


class MCP_ChatBot:

    def __init__(self):
        # Store active MCP sessions for all connected servers.
        self.sessions: List[ClientSession] = []

        # Manage and automatically close asynchronous connections and resources.
        self.exit_stack = AsyncExitStack()

        # Create the OpenAI client using environment variables.
        self.openai_client = OpenAI(api_key=openai_api_key, base_url=openai_base_url)

        # Store tools collected from all connected MCP servers.
        self.available_tools: List[ToolDefinition] = []

        # Map each tool name to the MCP session that provides it.
        self.tool_to_session: Dict[str, ClientSession] = {}

    async def connect_to_server(self, server_name: str, server_config: dict) -> None:
        """Connect to a single MCP server."""
        try:
            # Convert the server configuration into MCP stdio parameters.
            server_params = StdioServerParameters(**server_config)

            # Start the MCP server process and open its stdio transport.
            stdio_transport = await self.exit_stack.enter_async_context(
                stdio_client(server_params)
            )
            read, write = stdio_transport

            # Create an MCP client session using the server's input/output streams.
            session = await self.exit_stack.enter_async_context(
                ClientSession(read, write)
            )

            # Perform the MCP initialization handshake.
            await session.initialize()
            self.sessions.append(session)

            # Discover the tools exposed by this MCP server.
            response = await session.list_tools()
            tools = response.tools
            print(f"\nConnected to {server_name} with tools:", [t.name for t in tools])

            for tool in tools:
                # Remember which server session should execute this tool.
                self.tool_to_session[tool.name] = session

                # Convert the MCP tool schema into OpenAI function-calling format.
                self.available_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                })
        except Exception as e:
            print(f"Failed to connect to {server_name}: {e}")

    async def connect_to_servers(self):
        """Connect to all configured MCP servers."""
        try:
            # Load all MCP server definitions from the JSON configuration file.
            with open("server_config.json", "r") as file:
                data = json.load(file)

            servers = data.get("mcpServers", {})

            # Connect to each configured MCP server one by one.
            for server_name, server_config in servers.items():
                await self.connect_to_server(server_name, server_config)
        except Exception as e:
            print(f"Error loading server configuration: {e}")
            raise

    async def process_query(self, query):
        # Build a system prompt dynamically from the available MCP tools.
        tool_info = "\n".join(
            f"- {t['function']['name']}: {t['function']['description']}"
            for t in self.available_tools
        )
        system_prompt = (
            "You are a helpful assistant with access to these tools:\n"
            f"{tool_info}\n"
            "When greeted, introduce yourself and your capabilities."
        )

        # Start a new message history for the current user query.
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': query}
        ]

        # Let the model decide whether it needs to call one of the tools.
        response = self.openai_client.chat.completions.create(
            model='gpt-5.1',
            tools=self.available_tools,
            messages=messages
        )

        process_flag = True
        while process_flag:
            message = response.choices[0].message
            finish_reason = response.choices[0].finish_reason

            # Print the final answer when the model does not request a tool.
            if finish_reason == 'stop' or not message.tool_calls:
                print(message.content)
                process_flag = False

            elif finish_reason == 'tool_calls':
                # Preserve the assistant message containing the tool requests.
                messages.append(message)

                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)
                    tool_call_id = tool_call.id

                    print(f"Calling tool {tool_name} with args {tool_args}")

                    # Select the MCP session that owns the requested tool.
                    session = self.tool_to_session[tool_name]

                    # Execute the tool through the correct MCP server.
                    result = await session.call_tool(tool_name, arguments=tool_args)

                    # Combine text blocks returned by the MCP tool into one string.
                    result_text = "".join(
                        block.text for block in result.content
                        if hasattr(block, 'text')
                    )

                    # Add the tool result to the conversation using its call ID.
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call_id,
                        "content": result_text
                    })

                # Send the tool results back to the model so it can answer the user.
                response = self.openai_client.chat.completions.create(
                    model='gpt-4o',
                    tools=self.available_tools,
                    messages=messages
                )

                if response.choices[0].finish_reason == 'stop':
                    print(response.choices[0].message.content)
                    process_flag = False

    async def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")

        # Continue accepting user queries until the user enters "quit".
        while True:
            try:
                query = input("\nQuery: ").strip()

                if query.lower() == 'quit':
                    break

                await self.process_query(query)
                print("\n")
            except Exception as e:
                print(f"\nError: {str(e)}")

    async def cleanup(self):
        """Cleanly close all resources using AsyncExitStack."""

        # Close every MCP session and stdio process managed by the exit stack.
        await self.exit_stack.aclose()


async def main():
    # Create the chatbot, connect to all servers, and start the chat loop.
    chatbot = MCP_ChatBot()

    try:
        await chatbot.connect_to_servers()
        await chatbot.chat_loop()
    finally:
        # Always close MCP connections, even when an error occurs.
        await chatbot.cleanup()


if __name__ == "__main__":
    # Start the asynchronous chatbot application.
    asyncio.run(main())

Writing mcp_project/new_mcp_chatbot.py


## Running the MCP Chatbot

**Terminal Instructions**

- به پوشه `mcp_project` برو:
    - `cd mcp_project`
- پکیج‌های مورد نیاز را نصب کن (اگه قبلاً نصب نکردی):
    - `pip install openai python-dotenv mcp arxiv`
- فایل `server_config.json` را با محتوای زیر در پوشه `mcp_project` بساز (اگه نداری):
    ```json
    {
        "mcpServers": {
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", "."]
            },
            "research": {
                "command": "python",
                "args": ["research_server.py"]
            },
            "fetch": {
                "command": "uvx",
                "args": ["-y", "@modelcontextprotocol/server-fetch"]
            }
        }
    }
    ```
- چت‌بات را اجرا کن:
    - `python mcp_chatbot.py`
- برای خروج از چت‌بات، `quit` بنویس.


Make sure to interact with the chatbot. Here are some query examples:
- Fetch the content of this website: https://modelcontextprotocol.io/docs/concepts/architecture and save the content in the file "mcp_summary.md", create a visual diagram that summarizes the content of "mcp_summary.md" and save it in a text file
- Fetch deeplearning.ai and find an interesting term. Search for 2 papers around the term and then summarize your findings and write them to a file called results.txt

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

## Final Notes

You are encouraged to refactor the code of `MCP_ChatBot` if you'd like:

- how to connect to servers asynchronously
- what if tools from different servers have the same name
- revisit the attributes
  
And maybe any other idea you may think of.